<a href="https://colab.research.google.com/github/ravindyaparami/multi-agents-on-a-lie-group/blob/main/SE(3)_Rigid_Body_Control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Geometric Virtual Structure and Tracking Control for Multi Agent Systems on $SE(3)$


This section shows how to specilaize the right-invariant AGLES-PID control and synchronization framework for **rigid bodies evolving on the Special Euclidean group SE(3)**.  
Each agent represents a fully actuated rigid body with left-invariant kinetic energy.

---

## 1 Rigid-Body Model on SE(3)

For each agent $A_i$:

\begin{align}
g_i &=
\begin{bmatrix}
R_i & x_i \\
0 & 1
\end{bmatrix}
\in SE(3), \qquad
R_i\!\in SO(3),\;
x_i\!\in\mathbb{R}^3.
\end{align}


- **Lie algebra:**  
$\mathfrak{se}(3)=\{(\Omega_i,v_i)\}$ with  
$\widehat{\Omega}_i\in\mathfrak{so}(3)$, $v_i\in\mathbb{R}^3$.

- **Adjoint and coadjoint actions:**
\begin{align}
\operatorname{Ad}_{g_i}
\begin{bmatrix}\Omega\\v\end{bmatrix}
&=
\begin{bmatrix}
R_i\Omega \\ R_i v + (R_i\Omega)\times x_i
\end{bmatrix},\qquad
\operatorname{Ad}_{g_i}^*
\begin{bmatrix}\Pi\\p\end{bmatrix}
=
\begin{bmatrix}
R_i\Pi + x_i\times R_i p\\
R_i p
\end{bmatrix}.
\end{align}

- **Inertia tensor:**
\begin{align}
\mathbb{I}_i&=
\begin{bmatrix}
J_i & 0\\
0 & m_i I_3
\end{bmatrix},\qquad
\mathbb{I}_i:\mathfrak{se}(3)\!\to\!\mathfrak{se}(3)^*.
\end{align}

---

## 2 · Right-Invariant Dynamics

\begin{align}
\dot{g}_i &= \omega_i\cdot g_i, \qquad
\dot{\pi}_i = f_i^{e} + f_i^{u},
\end{align}
where

\begin{align}
\omega_i&=\mathbb{I}_i^{-1}\pi_i
=\begin{bmatrix}\Omega_i\\v_i\end{bmatrix},\quad
\pi_i&=\begin{bmatrix}\Pi_i\\p_i\end{bmatrix}.
\end{align}

For the corresponding **virtual body** $V_i$:

\begin{align}
\dot{g}_{v_i}&=\omega_{v_i}\cdot g_{v_i},\qquad
\dot{\pi}_{v_i}=f_{v_i}^{u},\quad f_{v_i}^{e}\equiv0.
\end{align}

---

## 3 · Tracking Objective

Each agent maintains a fixed relative pose  
\begin{align}
g_r(t)&=\bar g_i\, g_{v_i}(t),
\end{align}
where constant $\bar g_i\!\in\!SE(3)$ defines the desired offset.

---

## 4 · Right-Invariant Error

\begin{align}
e_i &= g_{v_i} g_i^{-1}
=
\begin{bmatrix}
R_e & x_e\\
0 & 1
\end{bmatrix},
\qquad
R_e = R_{v_i} R_i^\top,\;
x_e = x_{v_i} - R_e x_i.
\end{align}

**Error kinematics**

$$
\begin{aligned}
\dot{e}_i &= \omega_{e_i}\cdot e_i,\\
\omega_{e_i} &= \omega_{v_i}-\operatorname{Ad}_{e_i}\omega_i.
\end{aligned}
$$

**Error momentum**

$$
\pi_{e_i}
= \operatorname{Ad}_{g_i}^*\mathbb{I}_i\operatorname{Ad}_{g_{v_i}^{-1}}\omega_{e_i}
= \operatorname{Ad}_{e_i^{-1}}^*\pi_{v_i}-\pi_i.
$$

**Momentum derivative identity**

$$
\frac{d}{dt}\!\left(\operatorname{Ad}_{e_i^{-1}}^*\pi_{v_i}\right)
= \operatorname{Ad}_{e_i^{-1}}^*\dot{\pi}_{v_i}
+ \operatorname{ad}_{\omega_{e_i}}^*\operatorname{Ad}_{e_i^{-1}}^*\pi_{v_i}.
$$

Hence

$$
\dot{\pi}_{e_i}
= \operatorname{Ad}_{e_i^{-1}}^*\dot{\pi}_{v_i}
- \dot{\pi}_i
+ \operatorname{ad}_{\omega_{e_i}}^*\operatorname{Ad}_{e_i^{-1}}^*\pi_{v_i}.
$$

---

## 5 · AGLES–PID Control Law on SE(3)

Let $f_e:SE(3)\!\to\!\mathbb{R}$ be a **polar Morse potential**  
(e.g. $f_e(R,x)=\tfrac12 k_R\,\mathrm{tr}(I-R)
+\tfrac12 k_x\|x\|^2$).  
Define differential $df_e = \pi_e\cdot e$ and an integral term $\dot{\pi}_I=\pi_e$.

Control input for agent $i$:

$$
f_i^{u}
=
\Big(
\operatorname{Ad}_{e_i^{-1}}^*\dot{\pi}_{r_i}
+ \operatorname{ad}_{\omega_{e_i}}^*\operatorname{Ad}_{e_i^{-1}}^*\pi_{r_i}
- f_i^{e}
\Big)
- k_p\pi_{e_i}-k_d\pi_{e_i}-k_I\pi_{I_i}.
$$

Closed-loop **error dynamics**


\begin{align}
\dot{e}_i &= \omega_{e_i}\cdot e_i,\\
\dot{\pi}_{I_i} &= \pi_{e_i},\\
\dot{\pi}_{e_i} &= -k_p\pi_{e_i}-k_d\pi_{e_i}-k_I\pi_{I_i}.
\end{align}


> *The error dynamics do not get any simpler or more straightforward than this.*

---

## 6 · Virtual-System Synchronization

### Centralized
A global virtual frame $g_v(t)$ is broadcast; each $V_i$ tracks it via AGLES-PID.

### Decentralized
Each agent exchanges $(g_{v_i},\pi_{v_i})$ with neighbors and applies a consensus-based control $f_{v_i}^{u}$ to enforce  
$\lim_{t\to\infty} g_{v_i}(t)=g_v(t)$.


# Rigid Body PID Control

Conside a rigid body moving in space. Let $\mathbf{b}$ be a body fixed frame with the origin coinciding with that of the center of mass of the body, $M$ be the tatal mass of the body, $\mathbb{I}$ be the inertia tensor of the body with respect to the body frame $\mathbf{b}$, $f^e$ be the resultant total external interactions acting on the particles of the body, and $\tau^e$ be the total resultant of the moments of the external interactions about the center of mass of the body.

We have seen that a general rigid body is descibed by the following simple equations in an inertial frame $\mathbf{e}$ where the moments are taken about the center of mass of the rigid body.

\begin{align}
\dot{o}&=\frac{1}{M}p\\
\dot{R}&=\widehat{\omega}R,\\
\dot{p}&=f^e+f^u,\\
\dot{\pi}&=\tau^e+\tau^u,
\end{align}
where
\begin{align}
\omega &=(\mathbb{I}_R)^{-1}\pi,
\end{align}
Here we have split the force and control moments into unmanipulatable and manipulatable (control) part. The manipulatable (control) part will be denoted by a superscript $u$.

## The trajectory tracking problem

In this section we investigate the problem of ensuring asymptotic tracking of a sufficiently smooth trajectory $(o(t),R_r(t))$ in $\mathbb{R}^3\times SO(3)$.

Notice that the momentum equations are linear. Thus if the system is fully actuated all one needs is PID control.

However since the configuration space $\mathbb{R}^3\times SO(3)$ of rigid body motion is not a vector space defining the proportional and integral terms require some care.

Notice that the equations of motion are defined in the spatial momentum space. Thus the 'correct' way to define the configuration error and the integral term is to ensure that they are also spatial frame momentum quantities. We demonsrate below how this can be done.

### Configuration Error

We notice that the configuration space $\mathbb{R}^3\times SO(3)$ is a group. It is closed under the multiplication operation defined as follows. For
$(a_1,R_1),(a_2,R_2)\in \mathbb{R}^3\times SO(3)$
\begin{align}
(a_1,R_1)\cdot (a_2,R_2)&\triangleq \left((a_1-a_2),R_1R_2\right)
\end{align}
It has the unique inverse
\begin{align}
(a_1,R_1)^{-1}&\triangleq \left(-a_1,R_1^{-1}\right).
\end{align}

Now one can define the configuration error between a system configuration $(o,R)\in \mathbb{R}^3\times SO(3)$ and a reference configuration $(o_r,R_r)\in \mathbb{R}^3\times SO(3)$ in a global sense as  
\begin{align}
(o_e,R_e)\triangleq \left((o_r-o),R^{-1}R_r\right)\in \mathbb{R}^3\times SO(3),
\end{align}
or alternatively as
\begin{align}
(o_e,R_e)\triangleq \left((o_r-o),R_rR^{-1}\right)\in \mathbb{R}^3\times SO(3),
\end{align}

The former is defined as the *left-invariant* error while the latter is defined as the *right-invariant* error.

Notice that both these errors are in the configuration space $\mathbb{R}^3\times SO(3)$ and not in the spatial momentum space of $\mathbb{R}^3\times SO(3)$.

### Tracking Error dynamics

Let $(o(t),R_r(t))$ be a desired trajectory and $\omega_r(t)$ be such that $\widehat{\omega}_r(t)=\dot{R}_rR_r^T$. In line with the system define the reference spatial linear momentum $p_r\triangleq M\dot{o_r}$ and the spatial angular momentum $\pi_r\triangleq R_r\mathbb{I}R_r^T\omega_r$

Define the configurarion error
\begin{align}
o_e&=o_r-o,\\
R_e&=R_rR^T.
\end{align}
Then the spatial angular velcity of the tracking error is defined by
\begin{align}
\widehat{\omega}_e&\triangleq \dot{R}_eR_e^T=\widehat{\omega}_r(t)-R_e\widehat{\omega}(t)R_e^T
\end{align}
and hence that
\begin{align}
\omega_e&=\omega_r-R_e\omega.
\end{align}
Define the angular momentum error as
\begin{align}
\pi_e\triangleq R\mathbb{I}R_r^T\omega_e=R\mathbb{I}R_r^T(\omega_r-R_e\omega)=R\mathbb{I}R_r^T\omega_r-R\mathbb{I}R^T\omega=R_e^T\pi_r-\pi.
\end{align}
Similarly define the linear momentum error $p_e\triangleq p_r-p$.

Differentiating $\pi_e$ we have
\begin{align}
\dot{\pi}_e&=R_e^T(\dot{\pi}_r-\omega_e\times \pi_r)-\dot{\pi}=R_e^T(R_r\dot{\Pi}_r+(\omega_r-\omega_e)\times \pi_r)-\dot{\pi}=(R\dot{\Pi}_r+\omega\times \pi_r)-\dot{\pi}
\end{align}



Thus we have the error dynamics
\begin{align}
\dot{o}_e&=\frac{1}{M}p_e,\\
\dot{R}_e&=\widehat{\omega}_eR_e,\\
\dot{p}_e&=M\ddot{o}_r-f^e-f^u,\\
\dot{\pi}_e&=(R\dot{\Pi}_r+\omega\times \pi_r)-\tau^e-\tau^u
\end{align}

### Feedforward plus PID-Control

\begin{align}
\dot{e}_{I_o}&=e_o,\\
f^u&=M\ddot{o}_r+k_{P_o}e_o+k_{D_o}p_e+k_{I_o}e_{I_o}\\
\dot{e}_{I_R}&=e_R,\\
\tau^u&=(R\dot{\Pi}_r+\omega\times \pi_r)-\tau^e+k_{P_R}e_R+k_{D_R}\pi_e+k_{I_R}e_{I_R}
\end{align}

### Controlled Error Dynamics

\begin{align}
\dot{e}_{I_o}&=e_o,\\
\dot{o}_e&=\frac{1}{M}p_e,\\
\dot{p}_e&=-k_{P_o}e_o-k_{D_o}p_e-k_{I_o}e_{I_o},\\
\dot{e}_{I_R}&=e_R,\\
\dot{R}_e&=\widehat{\omega}_eR_e,\\
\dot{\pi}_e&=-k_{P_R}e_R-k_{D_R}\pi_e-k_{I_R}e_{I_R}
\end{align}

Here \begin{align}
o_e&=o_r-o,\\
R_e&=R_rR^T.
\end{align}
$\widehat{\omega}_r(t)=\dot{R}_rR_r^T$, $p_r\triangleq M\dot{o_r}$, $\pi_r\triangleq R_r\mathbb{I}R_r^T\omega_r$
\begin{align}
\omega_e&=\omega_r-R_e\omega,
\end{align}
\begin{align}
\pi_e\triangleq R\mathbb{I}R_r^T\omega_e,
\end{align}
and $p_e\triangleq p_r-p$.

In the section below we proceed to find the correct proportional control action $(e_o,e_R)$.

### Appropriate form of the Proportional Control Term

We notice that on the vectorspace $\mathbb{R}^n$ the proportional control is proportional to $e\in \mathbb{R}^n$ and that it can be thought of as $e=df$ where $f:\mathbb{R}^n\mapsto \mathbb{R}$ is the quatratic function defined by $f(e)=e^Te$. From a geometric point of view the derivative of $f$ given by $df=e$ is an element of the 'momentum' space of $\mathbb{R}^n$.


As shown below we reproduce this process on $\mathbb{R}^3\times SO(3)$. Let $(o_e,R_e) \in \mathbb{R}^3\times SO(3)$ be the configuration error. Define the *error function*
$f:\mathbb{R}^3\times SO(3)\mapsto \mathbb{R}$ as follows:
\begin{align}
f(o_e,R_e)&=\frac{1}{2}\left(o_e^To_e+\mathrm{trace} \left(K(I_{3\times 3}-R_e)\right)\right),
\end{align}
where
\begin{align}
K&=\mathrm{diag}\{\mathbb{K}_1,\mathbb{K}_2,\mathbb{K}_3\}.
\end{align}
This function has a minimum when $(o_e,R_e)=(0,I_{3\times 3})$.


Recall that there exisits $(\theta_e,n_e)$ such that
\begin{align}
R_e&=I_{3\times 3}+\sin{\theta_e}\,\widehat{n}_e+(1-\cos{\theta_e})\,\widehat{n}^2_e.
\end{align}
Thus we see that the error function takes the form
\begin{align}
f(o_e,R_e)&=\frac{1}{2}\left(o_e^To_e+2\alpha_n\sin^2{\left(\frac{\theta_e}{2}\right)}\right)
\end{align}
where
$\alpha_n\triangleq \frac{1}{2}\left((\mathbb{K}_2+\mathbb{K}_3)n_{e_1}^2+(\mathbb{K}_3+\mathbb{K}_1)n_{e_2}^2+(\mathbb{K}_1+\mathbb{K}_2)n_{e_3}^2\right)$.

Let us find the derivative of the function. Consider an arbitrary curve, $c(s)$ on $\mathbb{R}^3\times SO(3)$ that passes through $(o_e,R_e)$. That is let $c(s)=(o(s),R(s))\in\mathbb{R}^3\times SO(3)$ where
$c:[-\epsilon,\epsilon] \mapsto \mathbb{R}^3\times SO(3)$
be such that $(o(0),R(0))=(o,R)$. Then the tangent to this curve at $(o,R)$ is definded by $(o',\omega')$ where
\begin{align}
o'&\triangleq\left.\dfrac{d}{ds}\right|_{s=0}o_e(s),\\
\widehat{\omega}'&\triangleq \left(\left.\dfrac{d}{ds}\right|_{s=0}R_e(s)\right)R^T_e.
\end{align}
Then the derivative of the function $f$, that is denoted by $(e_o,e_R)$ is defined by  
\begin{align}
\langle e_o,o'\rangle&=\left.\dfrac{d}{ds}\right|_{s=0}\frac{1}{2}(o^T_e(s)o_e(s))=\langle o_e,o'\rangle,\\
\langle e_R,\omega'\rangle&=\left.\dfrac{d}{ds}\right|_{s=0}\frac{1}{2}\left(\mathrm{trace}(K(I_{3\times3}-R_e(s)))\right)=-\frac{1}{2}\left(\mathrm{trace}(K\widehat{\omega}'R_e)\right)=-\frac{1}{4}\left(\mathrm{trace}((R_eK-KR_e^T)\widehat{\omega}')\right)=\langle e_R,\omega'\rangle,\\
\end{align}
Thus $e_R$ is given by
\begin{align}
\widehat{e}_R&=\frac{1}{2}\left(R_eK-KR_e^T\right).
\end{align}


From $R_e=I_{3\times 3}+\sin{\theta_e}\,\widehat{n}_e+(1-\cos{\theta_e})\,\widehat{n}^2_e$ we thus see that
\begin{align}
\widehat{e}_R&=\frac{1}{2}\left(\sin{\theta_e}(\widehat{n}_eK+K\widehat{n}_e)+2\sin^2{\left(\frac{\theta_e}{2}\right)}(\widehat{n}_e^2K-K\widehat{n}_e^2)\right).
\end{align}
Thus we see that when $\theta_e=0$ or when $\theta_e=\pi$ and $n_e\in \{e_1,e_2,e_3\}$ the derivative of $f$ is zero.

From the above analysis we see that the function $f$ has four critical points $\{(0,I_{3\times 3}),(0,N_1),(0,N_2),(0,N_3)\}$ where $N_i$ is a rotation about $e_i$ by an angle $\pi$. Out of these $(0,I_{3\times 3})$ is a global minimum while $\{(0,N_1),(0,N_2),(0,N_3)\}$ are global maxima. One can show that this is the closest that we can come to the notion of a quadartic function on $\mathbb{R}^3\times SO(3)$.

In [ ]:
K_1, K_2, K_3, n_1, n_2, n_3=symbols('K_1, K_2, K_3, n_1, n_2, n_3')

In [ ]:
K=Matrix([[K_1,0,0],[0,K_2,0],[0,0,K_3]]);
hatN=Matrix([[0,-n_3,n_2],[n_3,0,-n_1],[-n_2,n_1,0]])